In [ ]:
!nvidia-smi

Thu May 21 19:33:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr  -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.3/495.3 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 16.8 MB/s eta 0:00:00


### Purpose of accelerate:
1. Ease of Mulit-Device Training: Whether you're using multiple GPUs or TPUs, accelerate makes it easier to scale your model across devices with minimal code changes.
2. Mixed Precision: It allows models to be trained using precision, which can speed up training and reduce memory usage.
3. Zero Redudancy Optimizer (ZeRO): Helps manage large models by efficiently splitting the model across multiple devices.
4. Offload to CPU/SSD: Useful for large models that may not fit entirely into GPU memory, by allowing parts of the model or optimizer to be offloaded to CPU or eve SSD.

In [ ]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 7.9 MB/s eta 0:00:00


In [ ]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
from datasets import load_dataset
import pandas as pd

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch

nltk.download("punkt")

ds = load_dataset("koushik7198/Samsung-samsum_processed")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from transformers import AutoTokenizer, PegasusForConditionalGeneration

model = PegasusForConditionalGeneration.from_pretrained("google/pegasus-xsum")
tokenizer = AutoTokenizer.from_pretrained("google/pegasus-xsum")

ARTICLE_TO_SUMMARIZE = (
    "PG&E stated it scheduled the blackouts in response to forecasts for high winds "
    "amid dry conditions. The aim is to reduce the risk of wildfires. Nearly 800 thousand customers were "
    "scheduled to be affected by the shutoffs which were expected to last through at least midday tomorrow."
)

inputs = tokenizer(ARTICLE_TO_SUMMARIZE, max_length=1024, return_tensors="pt")

# Generate Summary
summary_ids = model.generate(inputs["input_ids"])
tokenizer.batch_decode(summary_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

"California's largest electricity provider has turned off power to hundreds of thousands of customers."

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = "cuda:0" if torch.cuda.is_available() else "cpu"

device

'cuda:0'

### Fine Tuning

In [ ]:
model = "google/pegasus-xsum"
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForSeq2SeqLM.from_pretrained(model).to(device)

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
ds = load_dataset("koushik7198/Samsung-samsum_processed")

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 9032
    })
    validation: Dataset({
        features: ['input', 'output'],
        num_rows: 3872
    })
    test: Dataset({
        features: ['input', 'output'],
        num_rows: 3227
    })
})

In [ ]:
split_lengths = [len(ds[split]) for split in ds]

print(f"Split Lengths: {split_lengths}")
print(f"Features: {ds['train'].column_names}")
print("\nDialogue")
print(ds["test"][1]["input"])
print("\nSummary")
print(ds["test"][1]["output"])

Split Lengths: [9032, 3872, 3227]
Features: ['input', 'output']

Dialogue
Lita: Hi Jane
Jane: Hi Lita
Lita: How's your day?
Jane: Oh, it's ok but I have a terrible headache
Lita: I bet it's the girls
Jane: Sure, children are a blessing but sometimes I'd like to run a way
Lita: I know, my son is seven now but I remember when he was two or three
Jane: Hahaha
Lita: Is Virginia still sick?
Jane: A little but at least she's not crying all the time anymore
Lita: Thank God!
Jane: Tina was making fun of her yesterday, she called her "farty-poop"
Lita: Oh, that's cruel!
Jane: I know but I must admit that after 3 days of Virginia's bowel sickness it made me laugh
Lita: Tina is about to turn 5 years old, right?
Jane: Yes, next week, on Sunday

Summary
Jane has a terrible headache because of her children. Virginia is still a little sick. Tina called her "farty-poop" yesterday. Tina's turning 5 years old next week on Sunday.


### Preparing Data For Training For Sequene To Sequence Model

{
  'dialogue': "Hi! How are you?",
  'summary': "The speaker is asking how the other person is"
}

{
  'input_ids': [123, 456, 789, ...], # Token IDS for the dialogue
  'attention_mask': [1,1,1,...], # Attention mask for the input
  'labels': [321, 654, 987, ...] # Token IDs for the summary (target)
}

In [ ]:
def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(example_batch["input"], max_length=1024, truncation=True)
    target_encodings = tokenizer(example_batch["output"], max_length=128, truncation=True)

    return {
        "input_ids": input_encodings["input_ids"],
        "attention_mask": input_encodings["attention_mask"],
        "labels": target_encodings["input_ids"]
    }

In [ ]:
ds_pt = ds.map(convert_examples_to_features, batched=True)

Map:   0%|          | 0/9032 [00:00<?, ? examples/s]

Map:   0%|          | 0/3872 [00:00<?, ? examples/s]

Map:   0%|          | 0/3227 [00:00<?, ? examples/s]

In [ ]:
ds_pt["train"]

Dataset({
    features: ['input', 'output', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 9032
})

In [ ]:
# Training

from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    # evaluation_strategy="epoch", # Removed as it caused a TypeError. This is often due to an environment issue loading an incorrect transformers version.
    save_strategy="epoch",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
    num_train_epochs=1,
    # predict_with_generate=True, # Removed as it caused a TypeError. Similar to evaluation_strategy, likely an environment issue.
    fp16=True
)

In [ ]:
trainer = Trainer(model=model, args=training_args,
                  data_collator=seq2seq_data_collator,
                  train_dataset=ds_pt["train"],
                  eval_dataset=ds_pt["validation"])

In [ ]:
trainer.train()

Step,Training Loss
10,1.700876
20,1.479571
30,1.374658
40,1.227302
50,1.537457
60,1.476286
70,1.801412
80,1.595459
90,1.723205
100,1.719750


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9032, training_loss=1.54727496437735, metrics={'train_runtime': 3045.8221, 'train_samples_per_second': 2.965, 'train_steps_per_second': 2.965, 'total_flos': 3244733093511168.0, 'train_loss': 1.54727496437735, 'epoch': 1.0})

In [ ]:
# Evaluation
def generate_batch_sized_chunks(list_of_elements, batch_size):
    """split the dataset into smaller batches that we can process simultaneously
    Yield successive batch-sized chunks from list_of_elements."""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]

def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                               batch_size=16, device=device,
                               column_text="article",
                               column_summary="highlights"):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)):

        inputs = tokenizer(article_batch, max_length=1024,  truncation=True,
                        padding="max_length", return_tensors="pt")
        summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                         attention_mask=inputs["attention_mask"].to(device),
                         length_penalty=0.8, num_beams=8, max_length=128)
        '''parameter for length penalty ensures that the model does not generate sequences that are too long.'''
        # Finally, we dcode the generated texts,
        # replace the  token, and add the decoded texts with the references to the metric.
        decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                clean_up_tokenization_spaces=True)
               for s in summaries]
        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    #  Finally compute and return the ROUGE scores.
    score = metric.compute()
    return score

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [ ]:
import evaluate

rouge_metric = evaluate.load("rouge")
rouge_names = ['rouge1', 'rouge2', 'rougeL', 'rougeLsum']
#rouge_metric = load_metric('rouge')

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
# Check for out-of-range token IDs
sample = ds["test"][0]
inputs = tokenizer(sample["input"], max_length=1024, truncation=True,
                   padding="max_length", return_tensors="pt")

print("Max token ID:", inputs["input_ids"].max().item())
print("Vocab size:", model.config.vocab_size)
# Max token ID must be < vocab_size

Max token ID: 12334
Vocab size: 96103


In [ ]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

sample_text = ds["test"][0]["input"]
inputs = tokenizer(sample_text, max_length=1024, truncation=True,
                   padding="max_length", return_tensors="pt").to(device)

# Test generate on a single example
output = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    length_penalty=0.8,
    num_beams=8,
    max_length=128
)
print(tokenizer.decode(output[0], skip_special_tokens=True))

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
model_name = "google/pegasus-xsum"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

### Interpreting Good vd Bad Rouge Scores
1. Scores close to 1: This indicates a strong overlap between the generated summary and the reference summary, which is desirable in summarization tasks. E.g. F1-score of 0.7 or higher across metrics is generally considered good.
2. Scores between 0.5 and 0.7: Indicates moderate overlap. The summary might be capturinf key points but is likely missing some  structure or important information.
3. Scores below 0.5: Suggest a poor match between the generated and reference summaries. The model might be generating irrelevant or incomplete summaries that don't capture te key ideas well

In [ ]:
## Save model
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained("google/pegasus-xsum")
tokenizer = AutoTokenizer.from_pretrained("google/pegasus-xsum")


In [ ]:
# Load
tokenizer = AutoTokenizer.from_pretrained("google/pegasus-xsum")
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained("google/pegasus-xsum")

In [ ]:
gen_kwargs = {"length_penalty": 0.8, "num_beams":8, "max_length": 128}

sample_text = ds["test"][0]["dialogue"]

reference = ds["test"][0]["summary"]

pipe = pipeline("summarization", model=model_pegasus, tokenizer=tokenizer)

##
print("Dialogue:")
print(sample_text)

print("\nReference Summary:")
print(reference)